# 7주차 과제 베이스라인 — 심야약국 접근성 질문을 작게 나누기

심야약국 원본 출처는 아직 확정되지 않았습니다. 이번 주에는 분석 성공 여부보다 **시도·오류·문제 분해 기록**을 남기는 일이 더 중요합니다. 데이터가 없어도 아래 표와 기록 셀을 위에서부터 실행할 수 있습니다.

- ✅ **기본 시도**: 질문 하나를 고르고 요구사항표를 실행한 뒤 관찰과 다음 행동을 기록합니다.
- 🧩 **문제 분해**: 막힌 지점을 출처·운영시간·수요·좌표·거리 기준 중 하나로 좁힙니다.
- 🌱 **선택 탐색**: 데이터가 준비되면 약국 분석을 시작하거나, 완성된 CCTV 예제의 민감도 결과를 살펴봅니다.

**기본 완료 기준**은 질문 1개 + 실행 또는 실행 시도 1개 + 관찰한 결과 또는 오류 1개 + 다음 행동 1개입니다.


## 실행 전 준비 — 로컬 PC

아래 명령은 **노트북 셀이 아닌 터미널**에서 실행합니다. 먼저 `requirements.txt`가 있는 프로젝트 루트(`PSPY-DataAnalysis`)로 이동합니다. `.venv`는 패키지를 설치할 가상환경이고, `requirements.txt`는 루트에 이미 있는 설치 목록입니다.

**Windows PowerShell — 최초 한 번 생성하고 활성화**

```powershell
py -m venv .venv
.\.venv\Scripts\Activate.ps1
```

`py`가 없으면 생성 명령에 `python`을 사용합니다. 활성화가 실행 정책으로 차단되면 아래 공통 명령의 `python`을 `.\.venv\Scripts\python.exe`로 바꿉니다.

**macOS / Linux — 최초 한 번 생성하고 활성화**

```bash
python3 -m venv .venv
source .venv/bin/activate
```

**공통 — 같은 터미널에서 패키지 설치와 JupyterLab 실행**

```text
python -c "import sys; print(sys.executable)"
python -m pip install -r requirements.txt
python 00_OT/setup_folders.py
python 00_OT/check_environment.py
python -m ipykernel install --sys-prefix --name pspy-dataanalysis --display-name "Python (PSPY .venv)"
python -m jupyterlab
```

첫 명령의 경로가 프로젝트의 `.venv` 아래인지 확인합니다. JupyterLab에서 이 노트북을 열고 **Python (PSPY .venv)** 커널을 선택합니다. VS Code에서는 커널 선택에서 프로젝트의 `.venv` 파이썬을 고릅니다. 노트북에서도 `import sys; print(sys.executable)`로 같은 경로인지 확인할 수 있습니다. 과제 노트북도 같은 커널을 사용합니다.

새 터미널에서는 활성화부터 다시 진행하고 `python -m jupyterlab`을 실행합니다. 자세한 설명은 [OT의 로컬 가상환경 만들기](../00_OT/notion.md#로컬-가상환경-만들기)를 참고합니다.



## 실행 전 준비 — Colab과 공통 설정

로컬에서는 위 안내로 설치와 커널 선택을 마친 뒤 **아래 두 코드 셀부터 순서대로 실행**합니다. Colab에서는 다음 순서로 준비합니다.

1. Google Drive에 프로젝트 폴더 전체를 둡니다. `requirements.txt`, `07주차`, `dataset`의 상대 위치를 유지하고 이 노트북을 Colab으로 엽니다. 데이터는 본문의 출처에서 별도로 내려받아 프로젝트의 `dataset/open/extracted/` 아래 해당 데이터 폴더에 둡니다.
2. 아래 `COLAB_PROJECT_ROOT`를 자신의 Drive 프로젝트 경로로 수정합니다. 기본값은 `내 드라이브/Colab Notebooks/PSPY-DataAnalysis`입니다.
3. 첫 코드 셀을 실행해 Drive 연결을 허용합니다. Colab에서만 공통 패키지와 한글 폰트를 설치합니다. 로컬의 가상환경 생성·활성화는 필요하지 않습니다.
4. 두 번째 코드 셀에서 파이썬, 작업 폴더, 패키지 버전과 폰트를 확인합니다. 로컬 파이썬 경로는 프로젝트의 `.venv` 아래여야 합니다. 설치 후 Colab이 런타임 재시작을 요청하면 재시작한 뒤 준비 셀부터 다시 실행합니다.

두 환경 모두 작업 폴더를 `07주차`로 맞추므로 본문의 `../dataset/...` 경로를 그대로 사용할 수 있습니다. 이 준비 셀은 원본 데이터를 다운로드하지 않습니다. 파일이 필요한 단계에서는 본문의 데이터 준비 안내를 따릅니다.

선택형 HTML 지도를 만들 때만 새 코드 셀에서 `%pip install folium`을 실행합니다. 기본 공간분석과 과제 기록에는 필요하지 않습니다.


In [ ]:
# 로컬 / Colab 공통: 실행 환경과 데이터 상대경로 준비
from pathlib import Path
import os
import subprocess
import sys

WEEK_DIR_NAME = '07주차'
COLAB_PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/PSPY-DataAnalysis')

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True

if IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_ROOT = COLAB_PROJECT_ROOT
else:
    start = Path.cwd().resolve()
    PROJECT_ROOT = next(
        (path for path in [start, *start.parents]
         if (path / 'requirements.txt').is_file() and (path / WEEK_DIR_NAME).is_dir()),
        None,
    )

if PROJECT_ROOT is None or not (PROJECT_ROOT / 'requirements.txt').is_file():
    raise FileNotFoundError(
        'requirements.txt가 있는 프로젝트 루트를 찾지 못했습니다. '
        'Colab에서는 COLAB_PROJECT_ROOT를 수정하고, 로컬에서는 프로젝트 안에서 노트북을 여세요.'
    )

WEEK_DIR = PROJECT_ROOT / WEEK_DIR_NAME
if not WEEK_DIR.is_dir():
    raise FileNotFoundError(f'주차 폴더가 없습니다: {WEEK_DIR}. 프로젝트 폴더 전체를 준비하세요.')

if IN_COLAB:
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '-r', str(PROJECT_ROOT / 'requirements.txt')
    ])
    subprocess.check_call(['apt-get', 'update', '-qq'])
    subprocess.check_call(['apt-get', 'install', '-y', 'fonts-nanum'])
else:
    print('로컬 환경: Colab 설치를 건너뜁니다. 위 안내대로 .venv 커널을 선택하세요.')

os.chdir(WEEK_DIR)
print('실행 환경:', 'Colab' if IN_COLAB else '로컬')
print('프로젝트 루트:', PROJECT_ROOT)
print('작업 폴더:', Path.cwd())


In [ ]:
# 공통: 패키지, 커널, 한글 폰트 확인
import matplotlib
from matplotlib import font_manager
import pandas as pd
import sklearn

# Colab에서 방금 설치한 폰트를 현재 파이썬에도 등록합니다.
nanum_path = Path('/usr/share/fonts/truetype/nanum/NanumGothic.ttf')
if nanum_path.is_file():
    font_manager.fontManager.addfont(str(nanum_path))

available_fonts = {font.name for font in font_manager.fontManager.ttflist}
for font_name in ['Malgun Gothic', 'AppleGothic', 'NanumGothic']:
    if font_name in available_fonts:
        matplotlib.rcParams['font.family'] = font_name
        print('한글 폰트:', font_name)
        break
else:
    print('한글 폰트가 없습니다. 나눔고딕 설치 후 커널을 다시 시작하세요.')
matplotlib.rcParams['axes.unicode_minus'] = False

print('파이썬:', sys.executable)
print('pandas:', pd.__version__)
print('scikit-learn:', sklearn.__version__)
print('데이터 입력 폴더:', PROJECT_ROOT / 'dataset' / 'open' / 'extracted')


## ✅ 기본 시도 1 — 필요한 것을 표로 나눕니다

첫 셀은 데이터 파일을 읽지 않습니다. 먼저 어떤 자료와 기준이 필요한지 확인합니다.


In [ ]:
from pathlib import Path
import pandas as pd

requirements = pd.DataFrame([
    {'구분': '기존 시설', '필요한 내용': '심야 운영 약국의 이름·주소·위도·경도', '현재 상태': '공식 출처 미확정(TODO)'},
    {'구분': '수요 지점', '필요한 내용': '지역별 인구 또는 격자 중심 좌표', '현재 상태': '사용 자료 미확정(TODO)'},
    {'구분': '운영 기준', '필요한 내용': '심야로 볼 시간대와 요일', '현재 상태': '분석자가 먼저 정의할 수 있음'},
    {'구분': '거리 기준', '필요한 내용': '직선거리 단위와 커버 반경', '현재 상태': '예: m 단위, 1,000m'},
    {'구분': '결과', '필요한 내용': '질문에 답하는 표 또는 지도', '현재 상태': '데이터 준비 뒤 선택'},
])
requirements


표에 TODO가 남아 있어도 괜찮습니다. TODO가 **어떤 자료가 없어서 다음 계산으로 갈 수 없는지** 알려 주면 문제를 나눈 결과가 됩니다.

작은 힌트: 한 번에 출처·운영시간·좌표를 모두 해결하려 하지 말고, 현재 상태가 TODO인 행 하나만 골라 다음 행동을 정합니다.


## ✅ 기본 시도 2 — 질문 하나를 고릅니다


In [ ]:
question_options = pd.DataFrame({
    '질문 번호': [1, 2, 3],
    '질문': [
        '자정부터 오전 5시 사이에 운영하는 약국까지 직선거리가 먼 지역은 어디입니까?',
        '1km 안에 심야약국이 없는 수요 지점은 몇 곳입니까?',
        '수요가 많지만 심야약국 접근성이 낮을 가능성이 있는 지역은 어디입니까?',
    ],
})
question = question_options.loc[0, '질문']  # 다른 질문을 고르면 행 번호를 바꿉니다.
print('선택한 질문:', question)
question_options


## 🧩 문제 분해 — 지금 막힌 한 지점을 찾습니다

아래 셀은 현재 상태에서 다음에 확인할 일을 작은 단위로 보여 줍니다. 직접 확인한 내용이 생기면 `현재 관찰`을 고쳐 적습니다.


In [ ]:
breakdown = pd.DataFrame([
    {'순서': 1, '확인할 것': '공식 배포기관과 다운로드 링크', '현재 관찰': '미확정', '가장 작은 다음 행동': '배포기관 이름을 확인합니다.'},
    {'순서': 2, '확인할 것': '심야 운영시간 정의', '현재 관찰': '미정', '가장 작은 다음 행동': '사용할 시작·종료 시각을 적습니다.'},
    {'순서': 3, '확인할 것': '수요 자료의 한 행 의미', '현재 관찰': '미정', '가장 작은 다음 행동': '인구·격자·행정동 중 하나를 고릅니다.'},
    {'순서': 4, '확인할 것': '좌표 컬럼과 좌표계', '현재 관찰': '파일 준비 전', '가장 작은 다음 행동': '파일을 받은 뒤 컬럼명만 확인합니다.'},
    {'순서': 5, '확인할 것': '거리와 반경 단위', '현재 관찰': '직선거리 사용 예정', '가장 작은 다음 행동': 'm 또는 km 중 하나로 통일합니다.'},
])
breakdown


## ✅ 이번 주 학습 기록

아래 셀에는 이미 첫 시도 예시가 들어 있습니다. 실행한 뒤 실제로 확인한 내용에 맞게 문장을 한 곳 이상 고쳐 봅니다.


In [ ]:
attempt = '요구사항표와 질문 후보 표를 실행했습니다.'
observation = '심야약국 공식 출처와 수요 자료가 확정되지 않아 거리 계산 전 단계에서 멈췄습니다.'
next_action = '먼저 공식 배포기관과 심야 운영시간 기준을 확인하겠습니다.'

learning_log = pd.DataFrame({
    '기록 항목': ['질문', '실행 또는 실행 시도', '관찰한 결과 또는 오류', '다음 행동'],
    '내용': [question, attempt, observation, next_action],
})
learning_log


> **여기까지 하면 이번 주 기록 완료입니다.** 데이터 로딩이나 지도 작성에 성공하지 않아도 됩니다. 위 표를 발표 자료에 붙이고, 막힌 지점을 어떻게 나누었는지 짧게 설명합니다.


## 🌱 선택 탐색 A — 완성된 CCTV 결과를 활용합니다

`example.ipynb` Part 6을 실행했다면 생성된 민감도 CSV를 아래 셀로 확인할 수 있습니다. 파일이 없어도 오류로 멈추지 않고 다음 안내를 출력합니다. 반경 또는 신규 후보 수가 다른 두 행을 비교해 관찰 한 문장만 남겨도 됩니다.


In [ ]:
cctv_result_path = Path('../dataset/open/processed/서울시 CCTV 공공데이터/week7_sensitivity.csv')
if cctv_result_path.is_file():
    cctv_sensitivity = pd.read_csv(cctv_result_path)
    print('CCTV 민감도 결과:', cctv_sensitivity.shape)
    display(cctv_sensitivity.head())
else:
    print('민감도 CSV가 아직 없습니다.')
    print('07주차/example.ipynb Part 0~6을 실행하거나, notion.md의 저장된 민감도 표를 먼저 살펴봅니다.')


## 🌱 선택 탐색 B — 심야약국 데이터가 준비된 뒤 진행합니다

심야약국과 수요 데이터의 공식 출처는 아직 **TODO**입니다. 출처가 확정된 뒤에는 다음 중 하나만 골라 시도합니다.

- 파일의 행 수와 컬럼명을 출력합니다.
- 위도·경도 결측치 개수를 확인합니다.
- 중복 좌표 개수를 확인합니다.
- 수요 지점 한 곳과 약국 한 곳의 하버사인 거리를 계산합니다.
- 커버 반경 하나를 정하고 그 이유를 기록합니다.

작은 힌트: 처음부터 취약지역 3곳과 지도를 모두 만들 필요는 없습니다. 파일을 한 번 읽고 `shape`를 확인한 것만으로도 다음 브레이크다운을 시작할 수 있습니다.
